# Cleanup and Resources

CSC-239 · Module 7 · Lesson 3 of 4

Handling a failure is only part of finishing an operation. You must also release resources that the operation acquired. This lesson traces cleanup on normal completion, return, and failure.

Select the **Java** kernel in your Workspace. Start with a fresh kernel and run cells in order. This notebook creates its own starting state.


## Learning Goals

- Trace cleanup during normal work, return and an exception.
- Use try-with-resources and explain reverse close order and primary versus suppressed failures.


## Why This Matters

A program may open several files while processing a request. If work fails halfway through, those files still need to be closed so the program does not retain limited resources.


## Check Your Starting Point

Trace a thrown exception through a caller and its catch. Recall an interface, implements, @Override, a constructor, and an array traversal. Explain why returning from a method skips later ordinary statements in that method.

**My explanation:**


## Concept

### Give a resource a lifetime

A **resource lifetime** is the interval between acquiring a limited resource and releasing it. A file stream is one example: opening it begins use, and closing it ends that use. Keeping an unused resource open can prevent other work from proceeding or waste a limited operating-system resource.

This lesson uses small objects that print lifecycle messages. They model acquisition and cleanup; they do not open actual files or borrow real hardware. The next module applies the same structure to real file operations.

### Run cleanup as control leaves

A **finally block** normally runs when control leaves the associated try/catch. This includes normal completion, a return, or an exception. It runs after a matching catch, when one is present.

```java
try {
    System.out.println("Begin");
    throw new IllegalStateException("Work stopped.");
} catch (IllegalStateException problem) {
    System.out.println("Handled: " + problem.getMessage());
} finally {
    System.out.println("Cleanup");
}
System.out.println("After");
```

This prints Begin, Handled: Work stopped., Cleanup, and After on separate lines. IllegalStateException describes work that cannot proceed in its current state. The handler provides a response; finally performs cleanup before control continues after the statement.

A return also passes through finally:

```java
class CompletionTools {
    public static int finish() {
        try {
            System.out.println("Compute");
            return 7;
        } finally {
            System.out.println("Cleanup");
        }
    }
}
System.out.println("Result: " + CompletionTools.finish());
```

This prints Compute, Cleanup, and Result: 7. Java evaluates the return value, runs finally, and then completes that return.

Do not place a return or a new throw in finally just to finish cleanup. It can replace a pending return or failure and hide what really happened. Also, finally is not a guarantee against abrupt termination of the running Java program. Our lessons use ordinary control flow and do not forcibly terminate the running Java program.

### Use the automatic cleanup contract

The **AutoCloseable contract** is a Java interface with a close operation. A class implements it to make its resource cleanup available to automatic control flow. The close method should release what the object acquired.

A **try-with-resources statement** registers successfully initialized AutoCloseable objects for cleanup as its scope ends. A declaration appears inside parentheses after try:

```java
class NamedResource implements AutoCloseable {
    private String name;
    public NamedResource(String name) {
        this.name = name;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closed: " + name);
    }
}
try (NamedResource resource = new NamedResource("notes")) {
    System.out.println("Work");
}
System.out.println("Done");
```

This prints Opened: notes, Work, Closed: notes, and Done. The resource variable belongs to the try body. Java calls close when the body finishes. The print in this example lets us observe the call; a real resource class would perform actual cleanup there.

AutoCloseable belongs to java.lang, so it does not need an import. Its interface allows close to declare Exception. This simple implementation does not throw a checked exception, so its own close declaration can omit throws.

### Close multiple resources in reverse order

**Reverse resource closure** means that resources in one try-with-resources statement close in the opposite order from successful initialization. Separate the resource declarations with a semicolon. The worked example opens first, then second, so it closes second, then first.

This order is useful when later resources depend on earlier ones. The later resource finishes its cleanup while the earlier one is still available. If initialization of a later resource fails, Java still closes the earlier resources that were successfully initialized.

A failure in the body also triggers cleanup before an attached catch runs. Place your fingers on the output lines as you trace: acquire, acquire, body, close later, close earlier, catch. Do not put close calls only at the last ordinary statement of the body; a failure can skip that statement.

### Keep the primary failure visible

A **suppressed exception** is a secondary cleanup failure attached to an earlier primary failure. In try-with-resources, a failing close should not erase the body failure that first interrupted the operation.

```java
class FailingResource implements AutoCloseable {
    public FailingResource() {
        System.out.println("Opened");
    }
    @Override
    public void close() {
        System.out.println("Closing");
        throw new IllegalStateException("Close failed.");
    }
}
try (FailingResource resource = new FailingResource()) {
    throw new IllegalArgumentException("Body failed.");
} catch (IllegalArgumentException problem) {
    System.out.println("Primary: " + problem.getMessage());
    for (Throwable secondary : problem.getSuppressed()) {
        System.out.println("Suppressed: " + secondary.getMessage());
    }
}
```

This prints Opened, Closing, Primary: Body failed., and Suppressed: Close failed. The catch receives the original IllegalArgumentException. Its getSuppressed method returns an array of attached secondary failures. The enhanced for loop visits that array; an empty array would produce no Suppressed line.

A cause and a suppressed exception have different jobs. A cause records an earlier failure that led to a new translated exception. A suppressed exception records an additional failure while handling or closing around the primary failure. If the body succeeds but close fails, the close failure itself can become the escaping primary exception.


## Video Demonstration

Trace the resource declarations from left to right and the close operations from right to left. Predict the whole message sequence before the command runs.

<video controls preload="metadata" width="960">
  <source src="media/03_cleanup_and_resources/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/03_cleanup_and_resources/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the cleanup and resources demonstration transcript](media/03_cleanup_and_resources/transcript.md).


## Worked Example

**Subgoal 1: make cleanup observable.** NamedResource prints when construction and close occur.

**Subgoal 2: register both resources.** Declare first and second in one try-with-resources statement.

**Subgoal 3: compare body and cleanup order.** Print Work in the body and Done only after both close calls finish.


In [ ]:
class NamedResource implements AutoCloseable {
    private String name;
    public NamedResource(String name) {
        this.name = name;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closed: " + name);
    }
}
try (NamedResource first = new NamedResource("first");
     NamedResource second = new NamedResource("second")) {
    System.out.println("Work");
}
System.out.println("Done");


Expected output:

```text
Opened: first
Opened: second
Work
Closed: second
Closed: first
Done
```

Construction opens first and then second. After Work, Java closes second and then first. Done appears after both cleanup calls. The same automatic-close ordering also applies when the body throws, before a matching attached catch responds.


## Predict, Run, Trace, and Explain

### Predict the full resource lifetime

Read the complete program before running it. Predict every line for resources named "map" and "badge". Mark which lines come from constructors, the try body, close methods and the final statement. Explain what determines close order and whether Done can appear before either close call. Record your prediction before running the next cell or opening the answer.

My predicted complete output:

Which lines come from construction, body, close and the final statement:

What determines the close order:

Where each modeled resource lifetime begins and ends:


In [ ]:
class NamedResource implements AutoCloseable {
    private String name;
    public NamedResource(String name) {
        this.name = name;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closed: " + name);
    }
}
try (NamedResource first = new NamedResource("map");
     NamedResource second = new NamedResource("badge")) {
    System.out.println("Work");
}
System.out.println("Done");


Run the whole cell once. Keep your original prediction, compare every output line, and explain the first difference. Identify the two close calls and the statement that runs after both. Retain your corrected explanation. Run the complete program for each attempt so the class and its resource declarations are present.

My original prediction:

My actual complete output:

The first difference and why it occurred:

The two close calls in order:

Why Done is last:

My corrected post-run explanation:

### Trace return, failure and cleanup failures

Complete the lifetime table for the prediction program. Explain how implementing AutoCloseable makes close available and how the try-with-resources header arranges for Java to call it. Then trace the two supporting checks below. The first compares a return with a thrown failure passing through finally. The second distinguishes a primary body failure from a suppressed cleanup failure. Keep the original failure visible in your explanation rather than treating the cleanup message as a replacement.

| Resource | Construction position | Close position | Message that ends its modeled lifetime |
|---|---|---|---|
| map | | | |
| badge | | | |


What implements AutoCloseable promises:

What the try-with-resources statement does with these objects:

Why ordinary statements after a failing operation may be skipped:

My post-run explanation:

<details>
<summary>Show answer</summary>

The first constructor prints Opened: map, followed by Opened: badge from the second constructor. Both objects are successfully initialized in the try-with-resources header. Work prints in the body. When the body completes normally, Java calls close in reverse initialization order: badge, then map. Done appears after both calls finish. The name strings label the objects; they do not control the order. The AutoCloseable interface supplies the close contract, and these classes make the calls visible through print statements. They are classroom models and do not borrow actual items. The finally comparison and suppressed-failure model below extend this trace to other ways control leaves a block. A return or exception can skip later ordinary statements, so cleanup must be connected to leaving the protected work.

```java
class NamedResource implements AutoCloseable {
    private String name;
    public NamedResource(String name) {
        this.name = name;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closed: " + name);
    }
}
try (NamedResource first = new NamedResource("map");
     NamedResource second = new NamedResource("badge")) {
    System.out.println("Work");
}
System.out.println("Done");
```

Expected output:

```text
Opened: map
Opened: badge
Work
Closed: badge
Closed: map
Done
```

Common error: Using the name strings to decide close order. Closing resources in the same order as construction. Placing Done before automatic cleanup.

</details>


### Trace finally before a return or handler

Predict and then run both complete programs. The first calls finish(false); the second changes only the argument to true. For each, place Compute, Cleanup, the caller report and After completion in order. Explain when the return value reaches the caller and why the failure case has no Result line. Locate the finally block and the specific catch. Explain what makes cleanup run on these paths without claiming that it runs after abrupt termination of the running Java program.

My predicted and actual output for finish(false):

My predicted and actual output for finish(true):

When the return value becomes the caller's result:

Why the failed call has no Result line:

Why Cleanup comes before the caller's handler:

What these observations do and do not show about finally:

My post-run explanation:


**First program:**


In [ ]:
class CompletionTools {
    public static int finish(boolean fail) {
        try {
            System.out.println("Compute");
            if (fail) {
                throw new IllegalStateException("Computation stopped.");
            }
            return 9;
        } finally {
            System.out.println("Cleanup");
        }
    }
}
try {
    System.out.println("Result: " + CompletionTools.finish(false));
} catch (IllegalStateException problem) {
    System.out.println("Handled: " + problem.getMessage());
}
System.out.println("After completion");


**Comparison program:**


In [ ]:
class CompletionTools {
    public static int finish(boolean fail) {
        try {
            System.out.println("Compute");
            if (fail) {
                throw new IllegalStateException("Computation stopped.");
            }
            return 9;
        } finally {
            System.out.println("Cleanup");
        }
    }
}
try {
    System.out.println("Result: " + CompletionTools.finish(true));
} catch (IllegalStateException problem) {
    System.out.println("Handled: " + problem.getMessage());
}
System.out.println("After completion");


Record your own post-run explanation before opening the answer.

<details>
<summary>Show answer</summary>

With fail set to false, Compute prints and the method prepares to return 9. Before that return reaches the caller, finally prints Cleanup. The caller then prints Result: 9 and continues to After completion. With fail set to true, the body raises IllegalStateException before reaching the return. Finally still prints Cleanup as the failure leaves the method. The caller cannot complete its Result print; its specific catch prints Handled: Computation stopped. After completion follows. These are ordinary return and exception paths in a running Workspace; finally is not a guarantee against abrupt termination of the running Java program.

```java
class CompletionTools {
    public static int finish(boolean fail) {
        try {
            System.out.println("Compute");
            if (fail) {
                throw new IllegalStateException("Computation stopped.");
            }
            return 9;
        } finally {
            System.out.println("Cleanup");
        }
    }
}
try {
    System.out.println("Result: " + CompletionTools.finish(false));
} catch (IllegalStateException problem) {
    System.out.println("Handled: " + problem.getMessage());
}
System.out.println("After completion");
```

Expected output:

```text
Compute
Cleanup
Result: 9
After completion
```

Common error: Placing Result: 9 before Cleanup. Expecting the method to return 9 after it has thrown. Assuming finally only runs when the protected work fails.

**Check case 2.** The throw skips return 9, but leaving the try still runs finally. Cleanup prints before the exception reaches the caller's matching catch.

```java
class CompletionTools {
    public static int finish(boolean fail) {
        try {
            System.out.println("Compute");
            if (fail) {
                throw new IllegalStateException("Computation stopped.");
            }
            return 9;
        } finally {
            System.out.println("Cleanup");
        }
    }
}
try {
    System.out.println("Result: " + CompletionTools.finish(true));
} catch (IllegalStateException problem) {
    System.out.println("Handled: " + problem.getMessage());
}
System.out.println("After completion");
```

Expected output:

```text
Compute
Cleanup
Handled: Computation stopped.
After completion
```

</details>


### Keep a body failure and inspect cleanup failures

The supplied CleanupResource is a classroom model. Its failOnClose flag decides whether close raises an exception after printing its message. Predict the whole output before running. Identify which exception starts in the body, which starts during closing, and which one the catch receives. getSuppressed returns an array of secondary failures attached during cleanup; its length counts them, and the enhanced for loop reads each message. Explain why the map close still runs after the badge close fails. Then change only the badge constructor flag from true to false, predict and run the complete program, and explain the empty suppressed array. Restore the true flag. Distinguish this additional cleanup failure from an earlier cause retained inside a newly created exception.

My predicted complete output with the badge close failure:

My actual complete output:

The primary exception message and where it starts:

The suppressed message and where it starts:

The close order and why both calls are attempted:

My predicted and actual output when both close calls succeed:

Why an empty suppressed array prints no Suppressed lines:

How suppressed information differs from an earlier cause:

My post-run explanation:


In [ ]:
class CleanupResource implements AutoCloseable {
    private String name;
    private boolean failOnClose;
    public CleanupResource(String name, boolean failOnClose) {
        this.name = name;
        this.failOnClose = failOnClose;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closing: " + name);
        if (failOnClose) {
            throw new IllegalStateException("Close failed: " + name);
        }
    }
}
try (CleanupResource first = new CleanupResource("map", false);
     CleanupResource second = new CleanupResource("badge", true)) {
    System.out.println("Work");
    throw new IllegalStateException("Work failed.");
} catch (IllegalStateException problem) {
    System.out.println("Primary: " + problem.getMessage());
    Throwable[] secondary = problem.getSuppressed();
    System.out.println("Suppressed count: " + secondary.length);
    for (Throwable failure : secondary) {
        System.out.println("Suppressed: " + failure.getMessage());
    }
}
System.out.println("After cleanup");


Record your own post-run explanation before opening the answer.

<details>
<summary>Show answer</summary>

The resources open in the order map, badge. Work then raises the body exception with message Work failed. Automatic cleanup visits badge first; its close prints Closing: badge and raises another IllegalStateException. That later failure is attached to the original body exception as suppressed information. Java still calls close on map, which prints Closing: map and succeeds. The specific catch receives the primary body exception. getSuppressed returns an array with one attached cleanup failure, so the count is 1 and its message names badge. When both close flags are false, the body still fails and remains primary, but both cleanup calls succeed; the suppressed array is empty and the loop prints no Suppressed line. These are separate failure objects even though they have the same class. An earlier cause explains why a new exception was created; this suppressed exception records an additional failure during automatic cleanup.

```java
class CleanupResource implements AutoCloseable {
    private String name;
    private boolean failOnClose;
    public CleanupResource(String name, boolean failOnClose) {
        this.name = name;
        this.failOnClose = failOnClose;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closing: " + name);
        if (failOnClose) {
            throw new IllegalStateException("Close failed: " + name);
        }
    }
}
try (CleanupResource first = new CleanupResource("map", false);
     CleanupResource second = new CleanupResource("badge", true)) {
    System.out.println("Work");
    throw new IllegalStateException("Work failed.");
} catch (IllegalStateException problem) {
    System.out.println("Primary: " + problem.getMessage());
    Throwable[] secondary = problem.getSuppressed();
    System.out.println("Suppressed count: " + secondary.length);
    for (Throwable failure : secondary) {
        System.out.println("Suppressed: " + failure.getMessage());
    }
}
System.out.println("After cleanup");
```

Expected output:

```text
Opened: map
Opened: badge
Work
Closing: badge
Closing: map
Primary: Work failed.
Suppressed count: 1
Suppressed: Close failed: badge
After cleanup
```

Common error: Replacing the body message with the close failure message. Stopping the cleanup trace after badge close fails. Assuming an exception must have at least one suppressed failure. Using matching exception class names to conclude the two messages belong to the same object.

**Check case 2.** The body failure still reaches the catch after both resources close. No cleanup failure is attached, so getSuppressed returns an empty array and its traversal runs zero times.

```java
class CleanupResource implements AutoCloseable {
    private String name;
    private boolean failOnClose;
    public CleanupResource(String name, boolean failOnClose) {
        this.name = name;
        this.failOnClose = failOnClose;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closing: " + name);
        if (failOnClose) {
            throw new IllegalStateException("Close failed: " + name);
        }
    }
}
try (CleanupResource first = new CleanupResource("map", false);
     CleanupResource second = new CleanupResource("badge", false)) {
    System.out.println("Work");
    throw new IllegalStateException("Work failed.");
} catch (IllegalStateException problem) {
    System.out.println("Primary: " + problem.getMessage());
    Throwable[] secondary = problem.getSuppressed();
    System.out.println("Suppressed count: " + secondary.length);
    for (Throwable failure : secondary) {
        System.out.println("Suppressed: " + failure.getMessage());
    }
}
System.out.println("After cleanup");
```

Expected output:

```text
Opened: map
Opened: badge
Work
Closing: badge
Closing: map
Primary: Work failed.
Suppressed count: 0
After cleanup
```

</details>


## Guided Practice

Complete these tasks in order. The intentionally empty code cells are safe to run, but remain unfinished until you write and check your code.


### Complete the automatic cleanup structure

The displayed draft is incomplete and for reading only. Copy it into the empty work cell. Replace RESOURCE_CONTRACT, CLOSE_OPERATION and RESOURCE_SEPARATOR using `AutoCloseable`, `close` and `;`, each once. Keep the class fields, constructors, resource order, body and messages unchanged. Predict and run the completed program. Explain how the interface name, method name and separator each contribute to automatic cleanup.

This sample is for repair:

```java
class NamedResource implements RESOURCE_CONTRACT {
    private String name;
    public NamedResource(String name) {
        this.name = name;
        System.out.println("Opened: " + name);
    }
    @Override
    public void CLOSE_OPERATION() {
        System.out.println("Closed: " + name);
    }
}
try (NamedResource first = new NamedResource("map")RESOURCE_SEPARATOR
     NamedResource second = new NamedResource("badge")) {
    System.out.println("Work");
}
System.out.println("Done");
```


My three replacements and their jobs:

My predicted complete output:

My actual complete output:

What the interface requires:

Where both resources are registered for cleanup:

My post-run explanation:

<details>
<summary>Show answer</summary>

The first constructor prints Opened: map, followed by Opened: badge from the second constructor. Both objects are successfully initialized in the try-with-resources header. Work prints in the body. When the body completes normally, Java calls close in reverse initialization order: badge, then map. Done appears after both calls finish. The name strings label the objects; they do not control the order. The AutoCloseable interface supplies the close contract, and these classes make the calls visible through print statements. They are classroom models and do not borrow actual items. RESOURCE_CONTRACT is AutoCloseable, the interface the class implements. CLOSE_OPERATION is close, the required operation marked by @Override. RESOURCE_SEPARATOR is the semicolon between resource declarations inside the try parentheses. It separates map from badge in the initialization order; automatic closure uses the reverse order.

```java
class NamedResource implements AutoCloseable {
    private String name;
    public NamedResource(String name) {
        this.name = name;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closed: " + name);
    }
}
try (NamedResource first = new NamedResource("map");
     NamedResource second = new NamedResource("badge")) {
    System.out.println("Work");
}
System.out.println("Done");
```

Expected output:

```text
Opened: map
Opened: badge
Work
Closed: badge
Closed: map
Done
```

Common error: Using a comma between the resource declarations. Changing the required close method name. Moving one resource declaration outside the try header.

</details>


### Add a third resource and test failure

Add a third declaration, `NamedResource third = new NamedResource("pen")`, after badge inside the same try-with-resources header. Separate the declarations with semicolons. Keep the class and existing messages unchanged. Predict and run the complete three-resource program. Then add `throw new IllegalStateException("Label work stopped.");` after Work in the body and attach a specific catch after the resource statement that prints `Handled: ` plus the message. Keep Done after that handler. Predict and run again. Explain whether failure changes the close order or the position of the handler. Restore the normal three-resource body after testing.


In [ ]:
class NamedResource implements AutoCloseable {
    private String name;
    public NamedResource(String name) {
        this.name = name;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closed: " + name);
    }
}
try (NamedResource first = new NamedResource("map");
     NamedResource second = new NamedResource("badge")) {
    System.out.println("Work");
}
System.out.println("Done");


My three declarations in initialization order:

My predicted and actual normal output:

My predicted and actual body-failure output:

The close order on both paths:

Where the handler runs and why:

My post-run explanation and restored normal result:

<details>
<summary>Show answer</summary>

The declarations construct map, badge and pen in that order. After Work, automatic cleanup calls close on pen, then badge, then map. Done follows all three. In the failure variation, the body raises IllegalStateException after Work. All three close calls still run in the same reverse order before the attached catch prints Handled: Label work stopped. Done follows the handler. The extra resource extends the sequence without changing the cleanup rule.

```java
class NamedResource implements AutoCloseable {
    private String name;
    public NamedResource(String name) {
        this.name = name;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closed: " + name);
    }
}
try (NamedResource first = new NamedResource("map");
     NamedResource second = new NamedResource("badge");
     NamedResource third = new NamedResource("pen")) {
    System.out.println("Work");
}
System.out.println("Done");
```

Expected output:

```text
Opened: map
Opened: badge
Opened: pen
Work
Closed: pen
Closed: badge
Closed: map
Done
```

Common error: Adding the third object as an ordinary declaration in the body and assuming the header will close it. Closing the first-declared resource first. Placing the handler message before automatic closure.

**Additional test: `Three resources and a body failure`.** A body failure still triggers pen, badge and map cleanup before the specific handler responds. All three constructors had completed successfully.

```java
class NamedResource implements AutoCloseable {
    private String name;
    public NamedResource(String name) {
        this.name = name;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closed: " + name);
    }
}
try (NamedResource first = new NamedResource("map");
     NamedResource second = new NamedResource("badge");
     NamedResource third = new NamedResource("pen")) {
    System.out.println("Work");
    throw new IllegalStateException("Label work stopped.");
} catch (IllegalStateException problem) {
    System.out.println("Handled: " + problem.getMessage());
}
System.out.println("Done");
```

Expected output:

```text
Opened: map
Opened: badge
Opened: pen
Work
Closed: pen
Closed: badge
Closed: map
Handled: Label work stopped.
Done
```

</details>


### Repair cleanup skipped by a failure

The displayed draft calls close only through ordinary statements at the end of its try body. With stop set to true, predict the output and identify the missing cleanup lines. Keep this faulty draft in Markdown. Copy the whole program into the empty work cell, move the two resource declarations into a try-with-resources header, and remove the manual close calls. Keep the class, stop flag, body, exception and handler messages unchanged. Predict and run the repair. Then change only stop to false and run again. Explain why a matching catch alone did not ensure cleanup and why the repaired normal path should close each object exactly once.

This sample is for repair:

```java
class NamedResource implements AutoCloseable {
    private String name;
    public NamedResource(String name) {
        this.name = name;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closed: " + name);
    }
}
boolean stop = true;
try {
    NamedResource first = new NamedResource("map");
    NamedResource second = new NamedResource("badge");
    System.out.println("Work");
    if (stop) {
        throw new IllegalStateException("Label work stopped.");
    }
    second.close();
    first.close();
} catch (IllegalStateException problem) {
    System.out.println("Handled: " + problem.getMessage());
}
System.out.println("Done");
```


My predicted faulty output:

The missing close lines and the statement that skips them:

My repaired resource header:

My predicted and actual repaired output for stop true:

My predicted and actual repaired output for stop false:

Why the handler alone did not close the objects:

Why the old manual close calls must be removed:

My post-run explanation:

<details>
<summary>Show answer</summary>

In the faulty version, both constructors and Work run, then the conditional throw leaves the try body. The ordinary second.close and first.close statements are skipped. The catch reports the failure, but that response does not call either missing cleanup operation. The repair registers both initialized objects in a try-with-resources header. Java closes badge then map as control leaves the body, before the catch responds. Removing the old manual close calls prevents extra calls on the successful path. With stop false, there is no body exception; automatic cleanup still closes both once before Done.

```java
class NamedResource implements AutoCloseable {
    private String name;
    public NamedResource(String name) {
        this.name = name;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closed: " + name);
    }
}
boolean stop = true;
try (NamedResource first = new NamedResource("map");
     NamedResource second = new NamedResource("badge")) {
    System.out.println("Work");
    if (stop) {
        throw new IllegalStateException("Label work stopped.");
    }
} catch (IllegalStateException problem) {
    System.out.println("Handled: " + problem.getMessage());
}
System.out.println("Done");
```

Expected output:

```text
Opened: map
Opened: badge
Work
Closed: badge
Closed: map
Handled: Label work stopped.
Done
```

Common error: Keeping the old manual close calls after adding automatic cleanup, causing extra close messages on the normal path. Moving only one declaration into the resource header. Assuming the catch automatically executes skipped ordinary close statements.

**Additional test: `Repaired program with normal work`.** With stop false, the body completes and automatic cleanup runs once per resource in reverse order. Retaining manual closes would add unwanted duplicate Closed lines.

```java
class NamedResource implements AutoCloseable {
    private String name;
    public NamedResource(String name) {
        this.name = name;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closed: " + name);
    }
}
boolean stop = false;
try (NamedResource first = new NamedResource("map");
     NamedResource second = new NamedResource("badge")) {
    System.out.println("Work");
    if (stop) {
        throw new IllegalStateException("Label work stopped.");
    }
} catch (IllegalStateException problem) {
    System.out.println("Handled: " + problem.getMessage());
}
System.out.println("Done");
```

Expected output:

```text
Opened: map
Opened: badge
Work
Closed: badge
Closed: map
Done
```

</details>


## Independent Practice

### Build the equipment-return model

Write the complete `LoanResource` class and caller in the empty work cell. Implement `AutoCloseable` with a private String name, a constructor that stores its supplied name and prints `Borrowed: ` plus that name, and an overridden public void close method that prints `Returned: ` plus the name. This is a classroom model. Acquire `"camera"` and then `"tripod"` in one try-with-resources statement. Print `Begin work`, then throw `IllegalStateException` with message `Practice interruption.` Catch that specific type after automatic closure and print `Handled: ` plus its message. Print `Finished` once after the handler. Predict every line before running. Explain how the acquisition order determines the return order and why the handler follows both close calls. In the next stage, test normal work and a single resource, then adapt the supplied CleanupResource model to identify primary and suppressed messages.

My class and resource declarations:

My predicted complete output:

My actual complete output:

Where each modeled loan starts and ends:

Why tripod returns before camera:

Why the handler follows both return messages:

My post-run explanation:


### Check both work paths and retained cleanup failures

Test your complete LoanResource program in all four combinations below. For normal work, remove only the body throw; keep the specific handler in place. For a single resource, remove only the tripod declaration and keep camera. Predict the entire output before each run, record the actual lines, and check one Returned line per acquired object. Explain why a normal body has no Handled line and why Finished remains last. Restore the required two-resource failure case.

For a separate complete program, copy the earlier CleanupResource support, including its class and caller. Change the resource names to "camera" and "tripod", set both failOnClose flags to true, and change only the body exception message to "Practice interruption." Keep its Work, Closing, Primary, Suppressed count, Suppressed and After cleanup reports. Predict and run this adaptation. Identify the primary message, both suppressed messages and their order. Explain why the body failure stays primary and why both close attempts occur even though each raises an exception. Compare this with the earlier empty-suppressed-array case. Retain the original prediction, actual output and your corrected explanation for each test.

| LoanResource case | My predicted output | My actual output | Acquisition/return counts |
|---|---|---|---|
| camera + tripod; body fails | | | |
| camera + tripod; normal body | | | |
| camera only; body fails | | | |
| camera only; normal body | | | |


Why normal work has no Handled line:

Why each acquired object has one return line:

My restored baseline output:

My predicted and actual full CleanupResource adaptation output:

The primary message and where it starts:

The suppressed messages in order and why that order:

Why both close attempts occur:

How this differs from the empty suppressed array:

My corrections and post-run explanation:


<details>
<summary>Show answer</summary>

LoanResource implements the cleanup interface and prints observable acquisition/return messages. The caller constructs camera and then tripod before Begin work. The body throws Practice interruption. Automatic cleanup calls close on tripod first and camera second. Only after both calls finish does the matching handler print Handled: Practice interruption. Finished follows the handler. This preserves the original body failure while ensuring both successfully initialized classroom resources complete their modeled lifetimes. Removing the throw lets the body finish normally; both return messages still appear but the catch does not run. Keeping only camera produces one acquisition and one return on either work path. The separate CleanupResource adaptation gives both close calls a failure: tripod is attempted first, then camera. The body exception remains primary, while the two cleanup failures are attached in that closing order. The suppressed count is 2. In the earlier comparison with both close flags false, the body still failed but the count was 0, because no cleanup exception occurred.

```java
class LoanResource implements AutoCloseable {
    private String name;
    public LoanResource(String name) {
        this.name = name;
        System.out.println("Borrowed: " + name);
    }
    @Override
    public void close() {
        System.out.println("Returned: " + name);
    }
}
try (LoanResource first = new LoanResource("camera");
     LoanResource second = new LoanResource("tripod")) {
    System.out.println("Begin work");
    throw new IllegalStateException("Practice interruption.");
} catch (IllegalStateException problem) {
    System.out.println("Handled: " + problem.getMessage());
}
System.out.println("Finished");
```

Expected output:

```text
Borrowed: camera
Borrowed: tripod
Begin work
Returned: tripod
Returned: camera
Handled: Practice interruption.
Finished
```

Common error: Reversing the required camera-then-tripod construction order. Placing close calls only after the body throw. Printing the handler response before automatic cleanup. Changing the required messages or confusing the printed model with an actual equipment loan.

**Additional test: Two resources with normal work.** Removing the body throw leaves automatic reverse cleanup intact. The catch is skipped and Finished follows both return messages.

```java
class LoanResource implements AutoCloseable {
    private String name;
    public LoanResource(String name) {
        this.name = name;
        System.out.println("Borrowed: " + name);
    }
    @Override
    public void close() {
        System.out.println("Returned: " + name);
    }
}
try (LoanResource first = new LoanResource("camera");
     LoanResource second = new LoanResource("tripod")) {
    System.out.println("Begin work");
} catch (IllegalStateException problem) {
    System.out.println("Handled: " + problem.getMessage());
}
System.out.println("Finished");
```

Expected output:

```text
Borrowed: camera
Borrowed: tripod
Begin work
Returned: tripod
Returned: camera
Finished
```

**Additional test: Single camera resource with a body failure.** Only camera is acquired and automatically returned. Its close finishes before the specific handler prints the body failure.

```java
class LoanResource implements AutoCloseable {
    private String name;
    public LoanResource(String name) {
        this.name = name;
        System.out.println("Borrowed: " + name);
    }
    @Override
    public void close() {
        System.out.println("Returned: " + name);
    }
}
try (LoanResource first = new LoanResource("camera")) {
    System.out.println("Begin work");
    throw new IllegalStateException("Practice interruption.");
} catch (IllegalStateException problem) {
    System.out.println("Handled: " + problem.getMessage());
}
System.out.println("Finished");
```

Expected output:

```text
Borrowed: camera
Begin work
Returned: camera
Handled: Practice interruption.
Finished
```

**Additional test: Single camera resource with normal work.** Camera is acquired and returned once. Work succeeds, so there is no Handled line and Finished follows cleanup.

```java
class LoanResource implements AutoCloseable {
    private String name;
    public LoanResource(String name) {
        this.name = name;
        System.out.println("Borrowed: " + name);
    }
    @Override
    public void close() {
        System.out.println("Returned: " + name);
    }
}
try (LoanResource first = new LoanResource("camera")) {
    System.out.println("Begin work");
} catch (IllegalStateException problem) {
    System.out.println("Handled: " + problem.getMessage());
}
System.out.println("Finished");
```

Expected output:

```text
Borrowed: camera
Begin work
Returned: camera
Finished
```

**Additional test: Separate failing-close adaptation: camera and tripod both fail during cleanup.** The body failure Practice interruption. remains primary. Automatic closing attempts tripod and then camera, retaining both failures as suppressed in that order. The count of 2 and two distinct messages prove both close attempts were observed.

```java
class CleanupResource implements AutoCloseable {
    private String name;
    private boolean failOnClose;
    public CleanupResource(String name, boolean failOnClose) {
        this.name = name;
        this.failOnClose = failOnClose;
        System.out.println("Opened: " + name);
    }
    @Override
    public void close() {
        System.out.println("Closing: " + name);
        if (failOnClose) {
            throw new IllegalStateException("Close failed: " + name);
        }
    }
}
try (CleanupResource first = new CleanupResource("camera", true);
     CleanupResource second = new CleanupResource("tripod", true)) {
    System.out.println("Work");
    throw new IllegalStateException("Practice interruption.");
} catch (IllegalStateException problem) {
    System.out.println("Primary: " + problem.getMessage());
    Throwable[] secondary = problem.getSuppressed();
    System.out.println("Suppressed count: " + secondary.length);
    for (Throwable failure : secondary) {
        System.out.println("Suppressed: " + failure.getMessage());
    }
}
System.out.println("After cleanup");
```

Expected output:

```text
Opened: camera
Opened: tripod
Work
Closing: tripod
Closing: camera
Primary: Practice interruption.
Suppressed count: 2
Suppressed: Close failed: tripod
Suppressed: Close failed: camera
After cleanup
```

</details>


## Summary

Resource cleanup belongs to the resource lifetime. Finally normally runs as control leaves try/catch, even during a pending return. AutoCloseable and try-with-resources connect an object’s close operation to scope exit. Multiple resources close in reverse order. A body failure stays primary when a later close failure is suppressed.

Close the answers. Trace the normal, return, and failure paths, then explain the difference between a cause and a suppressed exception.


## Reflection

A task opens an input stream and then an output stream. Describe how cleanup should proceed if writing fails, and what information you need if closing also fails.

**My design and explanation:**

Next, you will write automated tests for successful results and expected failures.


## Supplemental Reading

- [Catching exceptions and cleanup](https://dev.java/learn/exceptions/catching-handling/) connects handlers, finally, and automatic resource management.
- [Java 21 try-with-resources rules](https://docs.oracle.com/javase/specs/jls/se21/html/jls-14.html#jls-14.20.3) specifies initialization, closure, and suppressed failures.
- [Java 21 AutoCloseable API](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/AutoCloseable.html) defines the cleanup interface.
- [Java 21 suppressed exceptions](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/Throwable.html#getSuppressed()) documents how to inspect secondary failures.
